In [1]:
import pandas as pd
import numpy as np
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)

from xgboost import XGBClassifier

In [2]:
df = pd.read_csv(
    "../data/processed/cleaned_data.csv"
)

print(df.shape)
df.head()

(200000, 20)


,project_id,project_type,land_area_hectares,number_of_affected_families,compensation_status,approval_timeline_days,legal_disputes_count,possession_status,rehabilitation_progress_pct,stakeholder_responsiveness,historical_performance_score,administrator_id,manager,location,altitude_m,latitude,longitude,delay_status,delay_days,risk_score
0,LA-IND-0000001,Highway,41.58,9,Assessment/Processing,247,1,Acquired,63.5,Low,61.9,ADM-0290,MGR-05091,"Nainital, Uttarakhand",1612.9,30.0668,78.640313,Delayed,248,65.2
1,LA-IND-0000002,Railway,53.72,8,Paid in Full,223,1,Pending,67.3,Low,45.2,ADM-0350,MGR-10763,"Salem, Tamil Nadu",433.6,11.1271,77.155395,Delayed,186,65.7
2,LA-IND-0000003,Power,15.50,5,Not Started,172,1,Acquired,69.7,Medium,53.6,ADM-0040,MGR-05135,"Nainital, Uttarakhand",1512.9,30.0668,78.864017,On Time,55,58.5
3,LA-IND-0000004,Irrigation,31.00,11,Disputed,195,1,Partially Acquired,55.5,Medium,70.6,ADM-1065,MGR-05648,"Kamrup Metropolitan, Assam",259.6,26.2006,92.495593,On Time,47,60.8
4,LA-IND-0000005,Highway,110.35,123,Disputed,394,1,Acquired,80.0,Medium,53.9,ADM-1090,MGR-06966,"Khordha, Odisha",362.2,20.9517,85.049115,Delayed,248,66.4


In [3]:
import joblib

imputer = joblib.load("../model/pickles/imputer.pkl")
categorical_imputer = joblib.load("../model/pickles/categorical_imputer.pkl")
feature_columns = joblib.load("../model/pickles/featurecolumn.pkl")
raw_input_columns = joblib.load("../model/pickles/raw_input_columns.pkl")
encode = encoder = joblib.load("../model/pickles/encoder.pkl")

print("Raw input columns:", raw_input_columns)
print("Feature columns:", feature_columns)

Raw input columns: ['project_type', 'land_area_hectares', 'number_of_affected_families', 'compensation_status', 'approval_timeline_days', 'legal_disputes_count', 'possession_status', 'rehabilitation_progress_pct', 'stakeholder_responsiveness', 'historical_performance_score', 'altitude_m', 'latitude', 'longitude']
Feature columns: ['land_area_hectares', 'number_of_affected_families', 'approval_timeline_days', 'legal_disputes_count', 'rehabilitation_progress_pct', 'historical_performance_score', 'altitude_m', 'latitude', 'longitude', 'project_type_Airport', 'project_type_Highway', 'project_type_Industrial', 'project_type_Irrigation', 'project_type_Other Infrastructure', 'project_type_Power', 'project_type_Railway', 'project_type_Urban Transport', 'project_type_Water Supply', 'compensation_status_Assessment/Processing', 'compensation_status_Disputed', 'compensation_status_Not Started', 'compensation_status_Paid in Full', 'compensation_status_Partially Paid', 'possession_status_Acquired', 

In [4]:
TARGET = "delay_status"

# Columns that should not be used for prediction
DROP_COLUMNS = [
    "project_id",
    "administrator_id",
    "manager",
    "location",
    "delay_days",
    "risk_score"
]

X = df.drop(
    columns=[TARGET] + DROP_COLUMNS,
    errors="ignore"
)

y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget percentage:")
print(y.value_counts(normalize=True) * 100)

X shape: (200000, 13)
y shape: (200000,)

Target distribution:
delay_status
On Time    136000
Delayed     64000
Name: count, dtype: int64

Target percentage:
delay_status
On Time    68.0
Delayed    32.0
Name: proportion, dtype: float64


In [5]:
y = y.map({
    "On Time": 0,
    "Delayed": 1
})

print("Target mapping:")
print("On Time = 0")
print("Delayed = 1")

print("\nTarget distribution:")
print(y.value_counts())

Target mapping:
On Time = 0
Delayed = 1

Target distribution:
delay_status
0    136000
1     64000
Name: count, dtype: int64


In [6]:
print("Imputer expects:", imputer.feature_names_in_)
print("\nCategorical imputer expects:", categorical_imputer.feature_names_in_)
print("\nEncoder expects:", encoder.feature_names_in_)

Imputer expects: ['land_area_hectares' 'number_of_affected_families'
 'approval_timeline_days' 'legal_disputes_count'
 'rehabilitation_progress_pct' 'historical_performance_score' 'altitude_m'
 'latitude' 'longitude']

Categorical imputer expects: ['project_type' 'compensation_status' 'possession_status'
 'stakeholder_responsiveness']

Encoder expects: ['project_type' 'compensation_status' 'possession_status'
 'stakeholder_responsiveness']


In [7]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['land_area_hectares', 'number_of_affected_families', 'approval_timeline_days', 'legal_disputes_count', 'rehabilitation_progress_pct', 'historical_performance_score', 'altitude_m', 'latitude', 'longitude']

Categorical features:
['project_type', 'compensation_status', 'possession_status', 'stakeholder_responsiveness']


C:\Users\pankaj\AppData\Local\Temp\ipykernel_27832\1904067735.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


### Split

In [8]:
print("First 10 X indices:")
print(X.index[:10].tolist())

print("\nFirst 10 X rows:")
print(X.head(10))

First 10 X indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

First 10 X rows:
      project_type  land_area_hectares  number_of_affected_families  \
0          Highway               41.58                            9   
1          Railway               53.72                            8   
2            Power               15.50                            5   
3       Irrigation               31.00                           11   
4          Highway              110.35                          123   
5            Power               53.01                           10   
6          Airport              189.41                          155   
7  Urban Transport               24.98                           31   
8     Water Supply              187.92                          108   
9  Urban Transport               26.09                           37   

     compensation_status  approval_timeline_days  legal_disputes_count  \
0  Assessment/Processing                     247                     1   
1

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (160000, 13)
X_test: (40000, 13)


In [10]:
print("\nTest indices:")
print(X_test.index[:10].tolist())


Test indices:
[1428, 168445, 80105, 20165, 57323, 33315, 6369, 173612, 123980, 13665]


In [11]:
numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object","string", "category"]
).columns.tolist()

X_train_num = imputer.transform(
    X_train[numerical_features]
)

X_test_num = imputer.transform(
    X_test[numerical_features]
)

X_train_cat = categorical_imputer.transform(
    X_train[categorical_features]
)

X_test_cat = categorical_imputer.transform(
    X_test[categorical_features]
)

X_train_cat_encoded = encoder.transform(X_train_cat)

X_test_cat_encoded = encoder.transform(X_test_cat)

c:\Machine learning\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
c:\Machine learning\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [12]:
y_train_encoded = y_train
y_test_encoded = y_test

In [13]:
import numpy as np

X_train_final = np.hstack([
    X_train_num,
    X_train_cat_encoded
])

X_test_final = np.hstack([
    X_test_num,
    X_test_cat_encoded
])

print("X_train_final:", X_train_final.shape)
print("X_test_final:", X_test_final.shape)
print("feature_columns:", len(feature_columns))

X_train_final: (160000, 30)
X_test_final: (40000, 30)
feature_columns: 30


In [14]:
assert X_train_final.shape[1] == len(feature_columns)
assert X_test_final.shape[1] == len(feature_columns)

print("✓ Feature dimensions match")

✓ Feature dimensions match


### Logistic Regression

In [15]:
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

logistic_model.fit(X_train_final, y_train_encoded)

c:\Machine learning\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",2000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default 

In [16]:
lr_pred = logistic_model.predict(X_test_final)

lr_prob = logistic_model.predict_proba(X_test_final)[:, 1]

### Random Forest

In [17]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_final,
    y_train_encoded
)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [18]:

rf_pred = rf_model.predict(X_test_final)

rf_prob = rf_model.predict_proba(
    X_test_final
)[:, 1]

In [19]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_final,
    y_train_encoded
)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [20]:
xgb_pred = xgb_model.predict(X_test_final)

xgb_prob = xgb_model.predict_proba(
    X_test_final
)[:, 1]

In [21]:
# Logistic Regression
lr_pred = logistic_model.predict(X_test_final)
lr_prob = logistic_model.predict_proba(X_test_final)[:, 1]


# Random Forest
rf_pred = rf_model.predict(X_test_final)
rf_prob = rf_model.predict_proba(X_test_final)[:, 1]


# XGBoost
xgb_pred = xgb_model.predict(X_test_final)
xgb_prob = xgb_model.predict_proba(X_test_final)[:, 1]

In [22]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

def evaluate_model(name, y_true, y_pred, y_prob):

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "F1 Score": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true, y_prob
        ),
        "PR-AUC": average_precision_score(
            y_true, y_prob
        )
    }

### success metrics

In [23]:
results = []

results.append(
    evaluate_model(
        "Logistic Regression",
        y_test_encoded,
        lr_pred,
        lr_prob
    )
)

results.append(
    evaluate_model(
        "Random Forest",
        y_test_encoded,
        rf_pred,
        rf_prob
    )
)

results.append(
    evaluate_model(
        "XGBoost",
        y_test_encoded,
        xgb_pred,
        xgb_prob
    )
)

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,Logistic Regression,0.830900,0.692991,0.846641,0.762149,0.920217,0.852303
1,Random Forest,0.839075,0.728277,0.792969,0.759247,0.916835,0.843328
2,XGBoost,0.844975,0.775809,0.725078,0.749586,0.920295,0.852676


In [24]:
print("===== Logistic Regression =====")
print(confusion_matrix(y_test_encoded, lr_pred))

print("\n===== Random Forest =====")
print(confusion_matrix(y_test_encoded, rf_pred))

print("\n===== XGBoost =====")
print(confusion_matrix(y_test_encoded, xgb_pred))

===== Logistic Regression =====
[[22399  4801]
 [ 1963 10837]]

===== Random Forest =====
[[23413  3787]
 [ 2650 10150]]

===== XGBoost =====
[[24518  2682]
 [ 3519  9281]]


In [25]:
results_df.sort_values(
    by="F1 Score",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,Logistic Regression,0.830900,0.692991,0.846641,0.762149,0.920217,0.852303
1,Random Forest,0.839075,0.728277,0.792969,0.759247,0.916835,0.843328
2,XGBoost,0.844975,0.775809,0.725078,0.749586,0.920295,0.852676


In [26]:
thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for threshold in thresholds:

    xgb_threshold_pred = (
        xgb_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_test_encoded,
            xgb_threshold_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test_encoded,
            xgb_threshold_pred,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_test_encoded,
            xgb_threshold_pred,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1 Score
0,0.10,0.546147,0.962969,0.696995
1,0.15,0.588086,0.942500,0.724260
2,0.20,0.622785,0.919922,0.742738
3,0.25,0.652822,0.892734,0.754158
4,0.30,0.679850,0.863516,0.760754
5,0.35,0.704383,0.831172,0.762543
6,0.40,0.727973,0.797813,0.761294
7,0.45,0.750730,0.763750,0.757184
8,0.50,0.775809,0.725078,0.749586
9,0.55,0.801680,0.685937,0.739306


In [27]:
best_threshold_row = threshold_df.loc[
    threshold_df["F1 Score"].idxmax()
]

best_threshold = best_threshold_row["Threshold"]

print("Best threshold:", best_threshold)
print("Precision:", best_threshold_row["Precision"])
print("Recall:", best_threshold_row["Recall"])
print("F1 Score:", best_threshold_row["F1 Score"])

Best threshold: 0.3500000000000001
Precision: 0.7043829449152542
Recall: 0.831171875
F1 Score: 0.762543004587156


In [28]:
best_threshold = 0.45

xgb_final_pred = (
    xgb_prob >= best_threshold
).astype(int)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test_encoded,
        xgb_final_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test_encoded,
        xgb_final_pred,
        target_names=["On Time", "Delayed"],
        zero_division=0
    )
)

Confusion Matrix:
[[23954  3246]
 [ 3024  9776]]

Classification Report:
              precision    recall  f1-score   support

     On Time       0.89      0.88      0.88     27200
     Delayed       0.75      0.76      0.76     12800

    accuracy                           0.84     40000
   macro avg       0.82      0.82      0.82     40000
weighted avg       0.84      0.84      0.84     40000



In [29]:
import joblib

best_threshold = 0.45

joblib.dump(
    xgb_model,
    "../model/model.pkl"
)

joblib.dump(
    best_threshold,
    "../model/threshold.pkl"
)

print("✓ model.pkl saved")
print("✓ threshold.pkl saved")

✓ model.pkl saved
✓ threshold.pkl saved


### Save model metadata

In [30]:
model_metadata = {
    "model_type": "XGBoost",
    "target": "delay_status",
    "classes": {
        0: "On Time",
        1: "Delayed"
    },
    "threshold": best_threshold,
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns
}

joblib.dump(
    model_metadata,
    "../model/model_metadata.pkl"
)

print("✓ model_metadata.pkl saved")

✓ model_metadata.pkl saved


In [31]:
print(
    df.groupby("stakeholder_responsiveness")["delay_status"]
      .value_counts(normalize=True)
      .unstack()
)

delay_status                 Delayed   On Time
stakeholder_responsiveness                    
High                        0.000714  0.999286
Low                         0.497796  0.502204
Medium                      0.053637  0.946363


In [32]:
print(
    pd.crosstab(
        df["stakeholder_responsiveness"],
        df["delay_status"],
        normalize="index"
    ) * 100
)

delay_status                  Delayed    On Time
stakeholder_responsiveness                      
High                         0.071412  99.928588
Low                         49.779560  50.220440
Medium                       5.363730  94.636270


In [33]:
print("Encoder categories:")

for feature, categories in zip(
    categorical_features,
    encoder.categories_
):
    print(feature, "->", categories)

Encoder categories:
project_type -> ['Airport' 'Highway' 'Industrial' 'Irrigation' 'Other Infrastructure'
 'Power' 'Railway' 'Urban Transport' 'Water Supply']
compensation_status -> ['Assessment/Processing' 'Disputed' 'Not Started' 'Paid in Full'
 'Partially Paid']
possession_status -> ['Acquired' 'Disputed/Stayed' 'Partially Acquired' 'Pending']
stakeholder_responsiveness -> ['High' 'Low' 'Medium']


In [34]:
print("Sample project:")
print(X_test.iloc[0])

Sample project:
project_type                    Urban Transport
land_area_hectares                        11.74
number_of_affected_families                  15
compensation_status                Paid in Full
approval_timeline_days                      226
legal_disputes_count                          1
possession_status                       Pending
rehabilitation_progress_pct                74.6
stakeholder_responsiveness                  Low
historical_performance_score               58.3
altitude_m                                364.9
latitude                                26.8467
longitude                             80.926357
Name: 1428, dtype: object


In [35]:
print("\nActual label:")
print(y_test.iloc[0])


Actual label:
0


In [36]:
y_prob = xgb_model.predict_proba(X_test_final)[:, 1]

result = pd.DataFrame({
    "stakeholder_responsiveness": X_test["stakeholder_responsiveness"].values,
    "actual": y_test.values,
    "probability": y_prob
})

print(
    result
    .groupby("stakeholder_responsiveness")["probability"]
    .agg(["mean", "median", "min", "max"])
)

                                mean    median       min       max
stakeholder_responsiveness                                        
High                        0.001023  0.000235  0.000018  0.052737
Low                         0.495960  0.485088  0.000139  0.999908
Medium                      0.054116  0.013874  0.000028  0.953029


In [37]:
print(
    df.groupby("stakeholder_responsiveness")["delay_days"].agg(
        ["count", "mean", "median", "min", "max"]
    )
)

                             count        mean  median  min   max
stakeholder_responsiveness                                       
High                          4201   36.097596    35.0    0   321
Low                         120441  174.437899    89.0    0  1214
Medium                       75358   49.499549    37.0    0   916


In [38]:
print(
    df.groupby("stakeholder_responsiveness")["risk_score"].agg(
        ["count", "mean", "median", "min", "max"]
    )
)

                             count       mean  median  min   max
stakeholder_responsiveness                                      
High                          4201  29.433278    28.9  2.9  61.7
Low                         120441  63.682646    65.0  8.9  99.0
Medium                       75358  48.519750    50.6  4.5  76.0


In [39]:
print(
    pd.crosstab(
        df["stakeholder_responsiveness"],
        df["delay_status"],
        normalize="index"
    ) * 100
)

delay_status                  Delayed    On Time
stakeholder_responsiveness                      
High                         0.071412  99.928588
Low                         49.779560  50.220440
Medium                       5.363730  94.636270


In [40]:
print(
    df.groupby("delay_status")[
        [
            "approval_timeline_days",
            "legal_disputes_count",
            "rehabilitation_progress_pct",
            "historical_performance_score",
            "land_area_hectares",
            "number_of_affected_families"
        ]
    ].mean()
)

              approval_timeline_days  legal_disputes_count  \
delay_status                                                 
Delayed                   298.266078              2.509891   
On Time                   218.415235              1.167816   

              rehabilitation_progress_pct  historical_performance_score  \
delay_status                                                              
Delayed                         52.050080                     46.712378   
On Time                         74.530534                     64.384115   

              land_area_hectares  number_of_affected_families  
delay_status                                                   
Delayed                93.760445                    70.594000  
On Time                50.573308                    28.577132  


In [41]:
print(
    df.groupby("stakeholder_responsiveness")[
        [
            "approval_timeline_days",
            "legal_disputes_count",
            "rehabilitation_progress_pct",
            "historical_performance_score"
        ]
    ].mean()
)

                            approval_timeline_days  legal_disputes_count  \
stakeholder_responsiveness                                                 
High                                    153.399191              0.243513   
Low                                     269.271320              2.064264   
Medium                                  208.574591              0.926391   

                            rehabilitation_progress_pct  \
stakeholder_responsiveness                                
High                                          89.751654   
Low                                           60.224378   
Medium                                        77.454639   

                            historical_performance_score  
stakeholder_responsiveness                                
High                                           80.861247  
Low                                            52.648170  
Medium                                         67.214304  


In [42]:
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

temp = df[["stakeholder_responsiveness", "delay_status"]].copy()

temp["stakeholder_responsiveness"] = (
    temp["stakeholder_responsiveness"]
    .map({
        "High": 0,
        "Low": 1,
        "Medium": 2
    })
)

temp["delay_status"] = (
    temp["delay_status"]
    .map({
        "On Time": 0,
        "Delayed": 1
    })
)

simple_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

simple_model.fit(
    temp[["stakeholder_responsiveness"]],
    temp["delay_status"]
)

print(
    classification_report(
        temp["delay_status"],
        simple_model.predict(
            temp[["stakeholder_responsiveness"]]
        )
    )
)

              precision    recall  f1-score   support

           0       0.68      1.00      0.81    136000
           1       0.00      0.00      0.00     64000

    accuracy                           0.68    200000
   macro avg       0.34      0.50      0.40    200000
weighted avg       0.46      0.68      0.55    200000



c:\Machine learning\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Machine learning\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Machine learning\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [43]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

X_temp = pd.get_dummies(
    df["stakeholder_responsiveness"],
    dtype=int
)

y_temp = df["delay_status"].map({
    "On Time": 0,
    "Delayed": 1
})

simple_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

simple_model.fit(X_temp, y_temp)

temp_prob = simple_model.predict_proba(X_temp)[:, 1]

result = pd.DataFrame({
    "stakeholder_responsiveness": df["stakeholder_responsiveness"],
    "probability": temp_prob
})

print(
    result.groupby("stakeholder_responsiveness")["probability"]
    .mean()
)

stakeholder_responsiveness
High      0.000714
Low       0.497796
Medium    0.053637
Name: probability, dtype: float64


XG_boost is having some issue with prediction on stakeholder low 

#### so using random forest instead


In [44]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train_final,
    y_train_encoded
)

rf_pred = rf_model.predict(X_test_final)

rf_prob = rf_model.predict_proba(
    X_test_final
)[:, 1]

In [45]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

print("Accuracy :", accuracy_score(y_test_encoded, rf_pred))
print("Precision:", precision_score(y_test_encoded, rf_pred))
print("Recall   :", recall_score(y_test_encoded, rf_pred))
print("F1 Score :", f1_score(y_test_encoded, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_test_encoded, rf_prob))
print("PR-AUC   :", average_precision_score(y_test_encoded, rf_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_encoded, rf_pred))

print("\nClassification Report:")
print(classification_report(
    y_test_encoded,
    rf_pred,
    target_names=["On Time", "Delayed"]
))

Accuracy : 0.839075
Precision: 0.7282772476142642
Recall   : 0.79296875
F1 Score : 0.7592474847589483
ROC-AUC  : 0.9168348245059743
PR-AUC   : 0.8433280960985918

Confusion Matrix:
[[23413  3787]
 [ 2650 10150]]

Classification Report:
              precision    recall  f1-score   support

     On Time       0.90      0.86      0.88     27200
     Delayed       0.73      0.79      0.76     12800

    accuracy                           0.84     40000
   macro avg       0.81      0.83      0.82     40000
weighted avg       0.84      0.84      0.84     40000



In [46]:
rf_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(rf_importance.head(15))

                              feature  importance
5        historical_performance_score    0.195756
2              approval_timeline_days    0.132168
4         rehabilitation_progress_pct    0.119692
28     stakeholder_responsiveness_Low    0.087277
29  stakeholder_responsiveness_Medium    0.070775
0                  land_area_hectares    0.062578
1         number_of_affected_families    0.059075
8                           longitude    0.049542
6                          altitude_m    0.045345
3                legal_disputes_count    0.041808
7                            latitude    0.034820
19       compensation_status_Disputed    0.014104
21   compensation_status_Paid in Full    0.007148
23         possession_status_Acquired    0.006855
10               project_type_Highway    0.006545


In [47]:
import joblib

joblib.dump(
    rf_model,
    "../model/pickles/model.pkl"
)

['../model/pickles/model.pkl']

In [48]:
threshold = 0.50

joblib.dump(
    threshold,
    "../model/pickles/threshold.pkl"
)

['../model/pickles/threshold.pkl']

In [49]:
model_metadata = {
    "model_type": "Random Forest",
    "target": "delay_status",
    "classes": {
        0: "On Time",
        1: "Delayed"
    },
    "threshold": threshold,
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns
}

joblib.dump(
    model_metadata,
    "../model/pickles/model_metadata.pkl"
)

['../model/pickles/model_metadata.pkl']

In [50]:
print("Notebook model classes:")
print(rf_model.classes_)

notebook_prob = rf_model.predict_proba(X_test_final)[:, 1]

print("\nNotebook probability by actual class:")
print(
    "On Time (0):",
    notebook_prob[y_test_encoded == 0].mean()
)

print(
    "Delayed (1):",
    notebook_prob[y_test_encoded == 1].mean()
)

Notebook model classes:
[0 1]

Notebook probability by actual class:
On Time (0): 0.19796862745098037
Delayed (1): 0.7117203125


In [51]:
print("\nNotebook RF metrics:")

notebook_pred = (
    notebook_prob >= 0.5
).astype(int)

print("Accuracy:", accuracy_score(y_test_encoded, notebook_pred))
print("ROC-AUC:", roc_auc_score(y_test_encoded, notebook_prob))


Notebook RF metrics:
Accuracy: 0.838725
ROC-AUC: 0.9168348245059743


In [52]:
print("Notebook X_test_final:")
print(X_test_final[:5])

Notebook X_test_final:
[[ 11.74      15.       226.         1.        74.6       58.3
  364.9       26.8467    80.926357   0.         0.         0.
    0.         0.         0.         0.         1.         0.
    0.         0.         0.         1.         0.         0.
    0.         0.         1.         0.         1.         0.      ]
 [ 39.81       6.       106.         1.       100.        73.4
    0.        29.0588    76.019336   0.         0.         0.
    0.         0.         0.         1.         0.         0.
    0.         0.         0.         1.         0.         0.
    0.         0.         1.         0.         0.         1.      ]
 [141.75      39.       436.         1.        67.8       45.4
  129.9       21.2787    81.580827   0.         0.         0.
    0.         0.         0.         1.         0.         0.
    1.         0.         0.         0.         0.         1.
    0.         0.         0.         0.         1.         0.      ]
 [ 66.72      27.      

In [53]:
print("First 10 X indices:")
print(X.index[:10].tolist())

print("\nFirst 10 X rows:")
print(X.head(10))

First 10 X indices:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

First 10 X rows:
      project_type  land_area_hectares  number_of_affected_families  \
0          Highway               41.58                            9   
1          Railway               53.72                            8   
2            Power               15.50                            5   
3       Irrigation               31.00                           11   
4          Highway              110.35                          123   
5            Power               53.01                           10   
6          Airport              189.41                          155   
7  Urban Transport               24.98                           31   
8     Water Supply              187.92                          108   
9  Urban Transport               26.09                           37   

     compensation_status  approval_timeline_days  legal_disputes_count  \
0  Assessment/Processing                     247                     1   
1